In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Cargar dataset
df = pd.read_csv('pib_banco_mundial_50.csv', sep=';')

# Revisar las primeras filas
df.head()

,Country Name,Country Code,2019,2020,2021,2022,2023
0,Argentina,ARG,1.492100e+12,5.090000e+11,2.781000e+11,7.350000e+11,1.361300e+12
1,Brazil,BRA,8.961000e+11,2.279500e+12,1.978400e+12,1.322700e+12,1.234500e+12
2,United States,USA,2.164700e+12,6.277000e+11,6.323000e+11,1.705900e+12,1.531200e+12
3,China,CHN,8.600000e+09,1.491000e+11,1.204200e+12,2.200000e+12,9.592000e+11
4,India,IND,8.261000e+11,1.537000e+12,2.098200e+12,2.001300e+12,1.486400e+12


In [10]:
# Fila con PIB mundial
world_row = df[df['Country Name'] == 'World']

# Datos de países (sin la fila 'World')
countries_df = df[df['Country Name'] != 'World']

# Transponer para tener años como filas
X = countries_df.set_index('Country Name')[['2019','2020','2021','2022','2023']].T
y = world_row[['2019','2020','2021','2022','2023']].T

# Renombrar columnas para evitar confusión
y.columns = ['World']

# Convertir a tipo numérico
X = X.apply(pd.to_numeric)
y = y.apply(pd.to_numeric)

X.head()
y.head()

,World
2019,5.857570e+13
2020,6.451670e+13
2021,6.172130e+13
2022,6.391170e+13
2023,6.948090e+13


In [12]:
# Crear el modelo
model = LinearRegression()

# Ajustar el modelo
model.fit(X, y)

# Predicciones
y_pred = model.predict(X)


In [14]:
# R²
r2 = r2_score(y, y_pred)
# Error cuadrático medio
mse = mean_squared_error(y, y_pred)
# Error absoluto medio
mae = mean_absolute_error(y, y_pred)

print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.2f}")
print(f"MAE: {mae:.2f}")

R²: 1.0000
MSE: 0.00
MAE: 0.00


Esto significa que el modelo predice el PIB mundial exactamente. Esto es completamente lógico en este caso, porque el PIB mundial (World) es la suma de todos los PIB de los países.

In [16]:
coefficients = pd.DataFrame({
    'Country': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

coefficients.head(10)  # Top 10 países que más influyen

,Country,Coefficient
0,Argentina,0.674939
21,Colombia,0.655892
13,South Korea,0.604752
48,Portugal,0.602970
36,Vietnam,0.481129
35,Thailand,0.428489
25,Bolivia,0.398859
20,Chile,0.391163
43,Finland,0.373660
6,France,0.370242


In [20]:
# Tomamos solo los países (sin 'World')
countries_df = df[df['Country Name'] != 'World']

# Calculamos la suma total por año
total_pib = countries_df[['2019','2020','2021','2022','2023']].sum()

# Calculamos la contribución relativa de cada país (porcentaje del total mundial)
contribuciones = countries_df.set_index('Country Name')[['2019','2020','2021','2022','2023']].div(total_pib) * 100

# Promedio de contribución de cada país en los 5 años
contribuciones['Promedio'] = contribuciones.mean(axis=1)

# Ordenamos de mayor a menor
contribuciones = contribuciones.sort_values(by='Promedio', ascending=False)

contribuciones.head(10)  # Top 10 países

,2019,2020,2021,2022,2023,Promedio
Country Name,,,,,,
Hungary,3.616687,2.422164,3.966054,3.302838,2.458086,3.153166
Venezuela,2.261347,3.457244,3.953254,3.388425,2.423400,3.096734
Canada,3.314173,3.733607,3.622574,1.934388,2.203771,2.961703
Netherlands,1.703949,3.680597,3.394776,2.911204,2.343090,2.806723
Paraguay,2.738337,2.020097,2.877613,3.777243,1.874760,2.657610
Pakistan,4.152234,2.756806,2.787530,0.265053,3.249094,2.642143
Mexico,3.322367,3.238851,1.468213,2.611572,2.138142,2.555829
Iran,3.435725,0.244433,2.846991,3.058751,3.032632,2.523706
India,1.410312,2.382329,3.399475,3.131352,2.139293,2.492552


Los resultados obtenidos muestran dos formas de analizar la influencia de los países en el PIB mundial. Por un lado, los coeficientes del modelo de regresión lineal reflejan la relación de cada país con el PIB mundial según el modelo. Por ejemplo, Argentina (0.6749), Colombia (0.6559) y South Korea (0.6047) son los países con mayor influencia, lo que significa que por cada unidad que aumenta su PIB, el PIB mundial aumenta aproximadamente en esa proporción. Sin embargo, estos coeficientes no reflejan directamente la contribución absoluta al PIB global, sino la influencia relativa dentro del modelo, que puede verse afectada por la escala o correlación entre países.

Por otro lado, las contribuciones promedio calculadas muestran el porcentaje real que cada país aporta al total del PIB mundial durante los años analizados. Por ejemplo, Hungary contribuye en promedio con un 3.15%, Venezuela con 3.09% y Canada con 2.96%. Este análisis indica qué países tienen mayor peso en términos absolutos en el PIB global.

Combinando ambos enfoques, podemos identificar los países que son determinantes tanto en influencia relativa (coeficientes altos) como en aporte absoluto (contribuciones altas). Para visualizarlo de manera clara, se podría generar un gráfico de dispersión donde el eje X represente la contribución promedio (%) y el eje Y los coeficientes del modelo, permitiendo identificar de un vistazo cuáles países son más relevantes para el PIB mundial.